In [5]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.desired_capabilities import DesiredCapabilities
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from bs4 import BeautifulSoup
import requests
import time
import pandas as pd
import re

In [6]:
columnas = ['Coche','Num_Inspecciones','Num_Registros','Tipo_Carroceria','Cilindrada (cm3)','Num_Puertas','Num_Asientos','Carga_Maxima (Kg)','Distancia_Ejes (mm)', 'Carfax_id']
df = pd.DataFrame(columns=columnas)

# Con Selenium (Muy Lento)

In [69]:
def carfax_scrapper(url):
    driver.get(url)
    coche = (driver.find_element(By.XPATH, '/html/body/div/div[1]/div[2]/div[2]/div[1]/div[1]/span')).text
    carroceria = (driver.find_element(By.XPATH, '/html/body/div/div[1]/div[2]/div[2]/div[1]/div[5]/span[2]')).text
    cilindrada = (driver.find_element(By.XPATH, '/html/body/div/div[1]/div[2]/div[2]/div[1]/div[9]/span[2]').text.split(' '))[0]
    asientos = (driver.find_element(By.XPATH, '//*[@id="section-content-technical-details"]/table[4]/tbody/tr[4]/td[2]/span')).text
    puertas = (driver.find_element(By.XPATH, '//*[@id="section-content-technical-details"]/table[4]/tbody/tr[3]/td[2]/span')).text
    carga_maxima = (driver.find_element(By.XPATH, '//*[@id="section-content-technical-details"]/table[2]/tbody/tr[2]/td[2]/span').text.split(' '))[0]
    distancia_ejes = (driver.find_element(By.XPATH, '//*[@id="section-content-technical-details"]/table[3]/tbody/tr/td[2]/span').text.split(' '))[0]
    cajas = driver.find_elements(By.CLASS_NAME, 'highlight-icon-box')
    for caja in cajas:
        if 'propietario' in caja.text:
            digito = re.findall(r'\d+', caja.text)
            propietarios = digito[0]
        if 'inspecciones' in caja.text:
            digito = re.findall(r'\d+', caja.text)
            inspecciones = digito[0]
        if 'registro' in caja.text:
            digito = re.findall(r'\d+', caja.text)
            registros = digito[0]
    carfax_id = hash(url)
    df.loc[len(df)] = [coche,inspecciones,registros,carroceria,cilindrada,puertas,asientos,carga_maxima,distancia_ejes, carfax_id]
    

# Con requests (Mucho más rápido)

In [7]:
def carfax_scrapper(url):
    request = requests.get(url)
    soup = BeautifulSoup(request.text, 'html.parser')
    coche = soup.find_all('div', class_='vdds-entry')[0].text.strip()
    carroceria = soup.find_all('div', class_='vdds-entry')[4].text.split()[-1]
    cilindrada = soup.find_all('div', class_='vdds-entry')[-4].text.split()[-2]
    num_puertas = soup.find_all('td', class_='technical-section-value-left')[-2].text.split()[0]
    num_asientos = soup.find_all('td', class_='technical-section-value-left')[-1].text.split()[0]
    carga_maxima = soup.find_all('td', class_="technical-section-value-right")[1].text.split()[0]
    distancia_ejes = soup.find_all('td', class_='technical-section-value-right')[-1].text.split()[0]
    propietarios = ''
    inspecciones = ''
    registros = ''
    cajas = soup.find_all('div', class_='highlight-icon-box')
    for caja in cajas:
        if 'propietario' in caja.text:
            digito = re.findall(r'\d+', caja.text)
            propietarios = digito[0]
        if 'inspecciones' in caja.text:
            digito = re.findall(r'\d+', caja.text)
            inspecciones = digito[0]
        if 'registro' in caja.text:
            digito = re.findall(r'\d+', caja.text)
            registros = digito[0]
    carfax_id = hash(url)
    df.loc[len(df)] = [coche,inspecciones,registros,carroceria,cilindrada,num_puertas,num_asientos,carga_maxima,distancia_ejes,carfax_id]
    

In [14]:
with open('links_OcasionPlus.txt', 'r') as enlaces:
    urls = enlaces.read().split()

In [17]:
for url in urls:
    try:
        request = requests.get(url)
        soup = BeautifulSoup(request.text, 'html.parser')
        informe = soup.find('li', class_='flex-row HistoryOption_historyOption__brd5W')
        informe = informe.find('a', class_='button_button__HOmVR button_link__am7Rj HistoryOption_itemBtn__EQq13')
        link = informe.get('href')
        carfax_scrapper(link)
        print(f'Coche añadido: {url}, Carfax: {link}')
    except:
        pass
df

Coche añadido: https://www.ocasionplus.com/coches-segunda-mano/toyota-auris-133-vvti-dual-active-con-155040km-2008-chemdqa5, Carfax: http://was.carfax.eu/icr/?dealer=1100093348&vinreg=NMTJM56E60R022539&checksum=b01ccc015ecc71ab363b1b5bcf231783
Coche añadido: https://www.ocasionplus.com/coches-segunda-mano/fiat-tipo-10-cross-con-68270km-2022-bpsvdqah, Carfax: http://was.carfax.eu/icr/?dealer=1100093348&vinreg=ZFACF7H35M6Y03660&checksum=57faca9cf5fb2eb89f7221936ddd2991
Coche añadido: https://www.ocasionplus.com/coches-segunda-mano/ford-focus-10-ecoboost-ss-titanium-con-165391km-2015-cb9inqal, Carfax: http://was.carfax.eu/icr/?dealer=1100093348&vinreg=WF05XXGCC5FJ03282&checksum=ddebe5b0cdf60ae053249ece0dc46545
Coche añadido: https://www.ocasionplus.com/coches-segunda-mano/volkswagen-troc-advance-10-tsi-con-92085km-2018-cipzqqal, Carfax: http://was.carfax.eu/icr/?dealer=1100093348&vinreg=WVGZZZA1ZJV049096&checksum=5f721d2337164b159a70410532a3e820
Coche añadido: https://www.ocasionplus.com/

,Coche,Num_Inspecciones,Num_Registros,Tipo_Carroceria,Cilindrada (cm3),Num_Puertas,Num_Asientos,Carga_Maxima (Kg),Distancia_Ejes (mm)
0,TOYOTA Auris,9,11,Compacto,1398,3,5,500,1.720
1,FIAT Tipo,,2,SUV/Monovolumen,999,5,5,500,2.638
2,FORD Focus,3,8,Compacto,999,5,5,695,2.648
3,VOLKSWAGEN T-Roc,2,3,Familiar,999,5,5,585,2.590
4,JEEP Renegade,,2,SUV/Monovolumen,999,5,5,540,2.570
...,...,...,...,...,...,...,...,...,...
1333,LAND ROVER Discovery,10,14,Manual,combustible:,5,7,725,2.720
1334,MINI One,3,9,Compacto,1198,3,4,480,2.495
1335,RENAULT MÃ©gane,6,10,Descapotable,1870,2,4,380,1.885
1336,LAND ROVER Discovery,1,4,Familiar,2993,5,5,752,2.922


In [18]:
df.to_csv('Carfax.csv', index=False)